# Dynamic Reality Modeling

In [1]:
import context
context.check_torch()

torch is installed:
GPU Name: NVIDIA GeForce RTX 5060 Ti
CUDA device count: 1
torch version: 2.8.0+cu128
torch-cuda version: 12.8


## Proposed Workflow

- Get partial scan and preferably Panoramic images or localised images.
- Object Detection
    - Perform 3D object detection on the colored pointcloud
- Scene completion
    - remove the bb from the scene and reconstruct the geometry
    - use the bb mask to inpaint on the pano image and reproject colors with new depthmap as guide
- Object completion
    - Reframe the pano image to center the detected object's bounding box
    - use the pano image as input image and the points as guide for the object completion network
- Dynamification
    - The resulting completed objects are 3D textures or Gaussians and can be modified in Unity

## Input

In [ ]:
%load_ext autoreload
%autoreload 2
import trimesh
from plyfile import PlyData, PlyElement
import numpy as np
from context import drm

pcdPath = r"/home/jvermandere/projects/DRM/_input/input_pc_scannet.ply"

In [ ]:
# read the raw ply data to extract the points
ply = PlyData.read(pcdPath)
plyData = ply['vertex'].data
splat = {name: np.asarray(plyData[name]) for name in plyData.dtype.names}
positions = np.stack([splat['x'], splat['y'], splat['z']], axis=1)

In [ ]:
drm.plot_points_3d(positions, up_axis='z', size=0.2, zoom=0.6)

## Object Detection


In [ ]:

from context import drm

### Votenet
We use Votenet to detect the different objects in a 3D scan, Votenet is located in another directory, so we have to link to it.

In [ ]:
import open3d as o3d
import trimesh

In [ ]:
pcdPath = r"/home/jvermandere/projects/DRM/_input/input_pc_scannet.ply"
pcd = o3d.io.read_point_cloud(pcdPath)
print(pcd)
o3d.visualization.draw(pcd)

In [ ]:
import trimesh
mesh = trimesh.load(pcdPath)
pct = trimesh.PointCloud(mesh.vertices)
trimesh.Scene(pct).show()

In [ ]:
import trimesh
pcdPath = r"/home/jvermandere/projects/DRM/_input/input_pc_scannet.ply"
boxSize = 0.01

# Load the PLY file as a mesh
mesh = trimesh.load(pcdPath)

# Extract points
if hasattr(mesh, 'vertices'):
    points = mesh.vertices
else:
    raise ValueError("PLY file does not contain vertices.")

# Create a cube at the origin
cube = trimesh.creation.box(extents=[boxSize, boxSize, boxSize])

# Translate cubes to each point
meshes = []
for p in points:
    m = cube.copy()
    m.apply_translation(p)
    meshes.append(m)

# Combine into a scene
scene = trimesh.Scene(meshes)

scene.show()



## 